In [2]:
!pip install pytubefix

  Using cached pytubefix-10.3.6-py3-none-any.whl.metadata (9.2 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
Using cached pytubefix-10.3.6-py3-none-any.whl (1.5 MB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   ------- -------------------------------- 7.3/41.3 MB 37.8 MB/s eta 0:00:01
   ----------- ---------------------------- 12.3/41.3 MB 30.8 MB/s eta 0:00:01
   ------------------------ --------------- 25.4/41.3 MB 41.3 MB/s eta 0:00:01
   ---------------------------------------  40.6/41.3 MB 49.7 MB/s eta 0:00:01
   ---------------------------------------- 41.3/41.3 MB 47.8 MB/s  0:00:00

   --- ------------------------------------  1/

In [9]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time
from collections import deque
from insightface.app import FaceAnalysis
from pytubefix import YouTube

YOUTUBE_URL = "https://youtu.be/cmdMopdk6lo"

TARGET_WIDTH = 800
FACE_PANEL_WIDTH = 150
FACE_SIZE = 120
MAX_FACES_DISPLAY = 4

DET_SIZE = (480, 480)
FONT = cv2.FONT_HERSHEY_SIMPLEX

START_TIME = 75    # วินาที
END_TIME = 120

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=DET_SIZE)

class FaceTracker:
    def __init__(self, iou_threshold=0.3, max_miss=15, traj_len=30):
        self.next_id = 1
        self.tracks = {}
        self.iou_threshold = iou_threshold
        self.max_miss = max_miss
        self.traj_len = traj_len
        self.colors = {}

    def _iou(self, a, b):
        x1,y1 = max(a[0],b[0]), max(a[1],b[1])
        x2,y2 = min(a[2],b[2]), min(a[3],b[3])
        inter = max(0,x2-x1)*max(0,y2-y1)
        areaA = (a[2]-a[0])*(a[3]-a[1])
        areaB = (b[2]-b[0])*(b[3]-b[1])
        union = areaA+areaB-inter
        return inter/union if union>0 else 0

    def _center(self, b):
        return ((b[0]+b[2])//2,(b[1]+b[3])//2)

    def _color(self, tid):
        if tid not in self.colors:
            hue = (tid*0.618)%1.0
            c = cv2.cvtColor(
                np.uint8([[[hue*180,255,255]]]),
                cv2.COLOR_HSV2BGR
            )[0][0]
            self.colors[tid] = tuple(map(int,c))
        return self.colors[tid]

    def update(self, boxes, faces):
        if boxes is None:
            for tid in list(self.tracks):
                self.tracks[tid]["miss"] += 1
                if self.tracks[tid]["miss"] > self.max_miss:
                    del self.tracks[tid]
            return self.tracks

        if len(self.tracks) == 0:
            for b,f in zip(boxes,faces):
                self._add(b,f)
            return self.tracks

        ids = list(self.tracks.keys())
        tboxes = [self.tracks[i]["box"] for i in ids]
        iou = np.zeros((len(boxes),len(tboxes)))

        for i,b in enumerate(boxes):
            for j,tb in enumerate(tboxes):
                iou[i,j] = self._iou(b,tb)

        used_d, used_t = set(), set()
        while True:
            if iou.size==0: break
            i,j = np.unravel_index(np.argmax(iou), iou.shape)
            if iou[i,j] < self.iou_threshold: break

            tid = ids[j]
            self.tracks[tid]["box"] = boxes[i]
            self.tracks[tid]["face"] = faces[i]
            self.tracks[tid]["miss"] = 0
            self.tracks[tid]["traj"].append(self._center(boxes[i]))

            used_d.add(i)
            used_t.add(j)
            iou[i,:] = -1
            iou[:,j] = -1

        for i,b in enumerate(boxes):
            if i not in used_d:
                self._add(b,faces[i])

        for j,tid in enumerate(ids):
            if j not in used_t:
                self.tracks[tid]["miss"] += 1
                if self.tracks[tid]["miss"] > self.max_miss:
                    del self.tracks[tid]

        return self.tracks

    def _add(self, box, face):
        self.tracks[self.next_id] = {
            "box": box,
            "face": face,
            "miss": 0,
            "traj": deque(maxlen=self.traj_len)
        }
        self.tracks[self.next_id]["traj"].append(self._center(box))
        self.next_id += 1

    def draw_traj(self, frame):
        for tid,d in self.tracks.items():
            if d["miss"]==0:
                pts = list(d["traj"])
                col = self._color(tid)
                for i in range(1,len(pts)):
                    cv2.line(frame, pts[i-1], pts[i], col, 2)
        return frame

def extract_face(frame, box, m=20):
    h,w = frame.shape[:2]
    x1,y1,x2,y2 = map(int,box)
    x1,y1 = max(0,x1-m), max(0,y1-m)
    x2,y2 = min(w,x2+m), min(h,y2+m)
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size else None

def create_face_panel(tracker, height):
    panel = np.ones((height, FACE_PANEL_WIDTH, 3), np.uint8)*40
    cv2.putText(panel,"Tracked",(10,25),FONT,0.5,(255,255,255),1)
    cv2.putText(panel,"Faces",(10,45),FONT,0.5,(255,255,255),1)

    y = 60
    active = [(tid,d) for tid,d in tracker.tracks.items()
              if d["miss"]==0 and d["face"] is not None]
    active.sort(key=lambda x:x[0])

    for tid,d in active[:MAX_FACES_DISPLAY]:
        if y+FACE_SIZE > height:
            break   # ⭐ FIX overflow

        face = cv2.resize(d["face"], (FACE_SIZE,FACE_SIZE))
        col = tracker._color(tid)

        panel[y:y+FACE_SIZE, 15:15+FACE_SIZE] = face
        cv2.rectangle(panel,(13,y-2),(15+FACE_SIZE+2,y+FACE_SIZE+2),col,2)
        cv2.putText(panel,f"ID:{tid}",(15,y-5),FONT,0.4,col,1)
        y += FACE_SIZE+10

    return panel

yt = YouTube(YOUTUBE_URL)
stream = yt.streams.filter(progressive=True, file_extension="mp4").first()
cap = cv2.VideoCapture(stream.url)

cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME*1000)

tracker = FaceTracker()

while cap.isOpened():
    clear_output(wait=True)
    ret, frame = cap.read()
    if not ret:
        break

    t = cap.get(cv2.CAP_PROP_POS_MSEC)/1000
    if t >= END_TIME:
        break

    faces = app.get(frame)
    boxes, crops = [], []

    for f in faces:
        boxes.append(f.bbox.astype(int))
        crops.append(extract_face(frame, f.bbox))

    tracker.update(boxes, crops)

    for tid,d in tracker.tracks.items():
        if d["miss"]==0:
            x1,y1,x2,y2 = d["box"]
            cv2.rectangle(frame,(x1,y1),(x2,y2),tracker._color(tid),2)
            cv2.putText(frame,f"ID:{tid}",(x1,y1-8),
                        FONT,0.6,(255,255,255),2)

    frame = tracker.draw_traj(frame)

    h,w = frame.shape[:2]
    nh = int(TARGET_WIDTH*h/w)
    frame = cv2.resize(frame,(TARGET_WIDTH,nh))

    panel = create_face_panel(tracker, nh)
    combined = np.hstack([frame, panel])

    _, buf = cv2.imencode(".jpg", combined)
    display(Image(data=buf.tobytes()))

    time.sleep(0.015)

cap.release()
print("Done")


KeyboardInterrupt: 